# Week 3 – Feast Feature Store for the IRIS pipeline
Roll no. 21f2000602 · Offline store: local Parquet (Task 6: BigQuery) · Online store: SQLite

In [1]:
%pip install "feast[gcp]" scikit-learn pandas joblib dvc dvc-gs

INFO: pip is looking at multiple versions of gcsfs to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of gcsfs to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
INFO: pip is looking at multiple versions of fsspec[tqdm] to determine which version is compatible with other requirements. This could take a while.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
INFO: pip is looking at multiple versions of pyopenssl to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versi

 - Checking Feast Version

In [1]:
!feast version

Feast SDK Version: "0.67.0"


### Initializing Feast

In [2]:
!feast init iris_feature_store


Creating a new Feast repository in /home/jupyter/21f2000602_MLOPS_WEEKLY_ASSIGNMENT/iris_feature_store.



In [3]:
!find iris_feature_store -type f -not -path "*/__pycache__/*"

iris_feature_store/README.md
iris_feature_store/feature_repo/feature_store.yaml
iris_feature_store/feature_repo/feature_definitions.py
iris_feature_store/feature_repo/test_workflow.py
iris_feature_store/feature_repo/__init__.py
iris_feature_store/feature_repo/data/driver_stats.parquet
iris_feature_store/feature_repo/data/driver_quality_labels.parquet
iris_feature_store/__init__.py
iris_feature_store/.gitignore


 - Checking Config file

In [4]:
!cat iris_feature_store/feature_repo/feature_store.yaml

project: iris_feature_store
# By default, the registry is a file (but can be turned into a more scalable SQL-backed registry)
registry: data/registry.db
# The provider primarily specifies default offline / online stores & storing the registry in a given cloud
provider: local
online_store:
    type: sqlite
    path: data/online_store.db
entity_key_serialization_version: 3
# By default, no_auth for authentication and authorization, other possible values kubernetes and oidc. Refer the documentation for more details.
auth:
    type: no_auth


- Explicitly write the oofline store config for later use

In [5]:
%%writefile -a iris_feature_store/feature_repo/feature_store.yaml
offline_store:
    type: file                   # offline (historical) store = local Parquet files via pandas

Appending to iris_feature_store/feature_repo/feature_store.yaml


In [6]:
!cat iris_feature_store/feature_repo/feature_store.yaml

project: iris_feature_store
# By default, the registry is a file (but can be turned into a more scalable SQL-backed registry)
registry: data/registry.db
# The provider primarily specifies default offline / online stores & storing the registry in a given cloud
provider: local
online_store:
    type: sqlite
    path: data/online_store.db
entity_key_serialization_version: 3
# By default, no_auth for authentication and authorization, other possible values kubernetes and oidc. Refer the documentation for more details.
auth:
    type: no_auth
offline_store:
    type: file                   # offline (historical) store = local Parquet files via pandas


#### (Optional) Removing Unnecessary files

In [7]:
!cd iris_feature_store/feature_repo && rm -f feature_definitions.py example_repo.py test_workflow.py data/driver_*.parquet
!rm -f iris_feature_store/README.md
!find iris_feature_store -type f -not -path "*/__pycache__/*"

iris_feature_store/feature_repo/feature_store.yaml
iris_feature_store/feature_repo/__init__.py
iris_feature_store/__init__.py
iris_feature_store/.gitignore


#### Keep Generated files out of git

In [8]:
%%writefile iris_feature_store/feature_repo/.gitignore
# Generated by Feast - rebuilt by `feast apply` / `feast materialize`, never committed
data/registry.db
data/online_store.db
# Parquet copy of the dataset - regenerated by the notebook from the DVC-tracked CSV
data/*.parquet

Writing iris_feature_store/feature_repo/.gitignore
